# Part 2 - NumPy for Performance

In Part 1 you saw that NumPy lets you operate on entire arrays without writing a loop. This notebook explains **why** that matters for performance, and introduces **broadcasting** and basic **linear algebra** - concepts that carry directly into GPU computing later today.

## Vectorization vs. Python Loops
A Python `for` loop processes one element at a time, with a lot of overhead per step. NumPy operations are implemented in compiled C code and operate on whole arrays at once (**vectorization**). Let's measure the difference on one million values.

In [ ]:
import numpy as np
import time

n = 1_000_000
values = list(range(n))
array_values = np.arange(n)

In [ ]:
# Pure Python loop
start = time.time()
squared_loop = [v * v for v in values]
loop_time = time.time() - start
print(f"Python loop: {loop_time:.4f} seconds")

In [ ]:
# NumPy vectorized operation
start = time.time()
squared_numpy = array_values ** 2
numpy_time = time.time() - start
print(f"NumPy vectorized: {numpy_time:.4f} seconds")

print(f"NumPy was about {loop_time / numpy_time:.0f}x faster")

This speedup is the reason scientific Python code avoids explicit loops over array data whenever possible - a rule of thumb sometimes called **"vectorize, don't iterate."** The same principle is what makes GPU computing (Notebook 5) so powerful: a GPU applies the same operation to thousands of elements simultaneously.

## Broadcasting
**Broadcasting** is NumPy's set of rules for performing operations on arrays of different shapes. The simplest case - an operation between an array and a single number - broadcasts the number across every element (you've already used this: `stress_array / 1000`).

Broadcasting also works between arrays of different shapes, as long as their dimensions are compatible.

In [ ]:
# A 3x3 grid of stress measurements (e.g., 3 samples x 3 test points)
stress_grid = np.array([
    [100, 150, 200],
    [110, 160, 210],
    [105, 155, 205],
])

# A per-sample correction factor (one value per row)
correction = np.array([1.0, 1.05, 0.98]).reshape(3, 1)

corrected = stress_grid * correction
print(corrected)

Broadcasting "stretches" the smaller array across the larger one without actually copying data, which keeps operations fast and memory-efficient.

## Basic Linear Algebra
NumPy's `@` operator (or `np.matmul`) performs matrix multiplication - a core operation in simulations, machine learning, and finite-element style calculations.

In [ ]:
# A simple stiffness matrix and a displacement vector
stiffness = np.array([
    [2.0, -1.0],
    [-1.0, 2.0],
])
displacement = np.array([0.5, 0.2])

force = stiffness @ displacement
print(f"Resulting force vector: {force}")

# Other useful linear algebra tools
print(f"Determinant: {np.linalg.det(stiffness):.2f}")
print(f"Inverse:\n{np.linalg.inv(stiffness)}")

## Views vs. Copies
Slicing a NumPy array usually returns a **view** - it shares memory with the original array. Modifying a view modifies the original! Use `.copy()` when you need an independent array.

In [ ]:
original = np.array([1, 2, 3, 4, 5])
view = original[1:3]
view[0] = 999
print("Original after modifying the view:", original)

original = np.array([1, 2, 3, 4, 5])
copy = original[1:3].copy()
copy[0] = 999
print("Original after modifying the copy:", original)

### *Exercise*
1. Time how long it takes to compute the element-wise square root of a one-million-element array using a Python loop vs. `np.sqrt()`.
2. Given a `(5, 3)` array of stress measurements, and a `(3,)` array of per-column calibration factors, use broadcasting to apply the calibration to every row.
3. Explain (in a markdown cell) why a view can cause subtle bugs when passing array slices into functions.

In [ ]:
# Enter your code here